# El valor de la información: prueba empírica y simulación

Sección 7 de la tesis.

**Prueba (Sección 7.5).** Se estima el modelo de estructura latente
$P(G_i = 1 \mid u) = \Lambda(\alpha + \ell_i + u_{k(i)})$, $u_k \sim N(0, \sigma^2)$,
como un logit de efectos aleatorios sobre los 247 sitios con resultado, con *offset* $\ell_i$ igual al logit del puntaje fuera de municipio del GLM. La verosimilitud integra $u_k$ por cuadratura de Gauss-Hermite. Se contrasta $\sigma^2 = 0$ con una razón de verosimilitudes cuya distribución, al estar el parámetro en la frontera, es $\tfrac12\chi^2_0 + \tfrac12\chi^2_1$ (Self y Liang, 1987). Zonas operativas $k(i)$: (a) municipio; (b) municipio × período (primera diligencia antes de 2010, desde 2010 o sin año); (c) fuente de la información del registro en las matrices de EQUITAS, como aproximación al proceso o campaña de documentación que llevó a la diligencia (los sitios sin fuente forman su propia zona). Además, se calcula el índice de Moran de los residuos $D_i - \hat p_i$ con los 5 vecinos más cercanos.

**Simulación (Sección 7.7).** Se compara el portafolio estático, una política miope de un paso por probabilidad de hallazgo por costo con actualización bayesiana, y la política de la Sección 7.4, que elige el sitio que maximiza $p_i(\mathcal D) + E_{y_i}[H_{-i}(b - c_i \mid \mathcal D \cup \{(i, y_i)\})]$.


In [ ]:
import glob
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import optimize, stats
from scipy.special import expit, logit
import warnings
warnings.filterwarnings('ignore')

import libpysal
from esda.moran import Moran

PATH_PROCESSED = Path("../data/Processed/")
PATH_MATRICES = "/Volumes/Backup Plus/Equitas/MIG/"
N_CUADRATURA = 40
SEMILLA = 42

pred = pd.read_csv(PATH_PROCESSED / "predicciones_fuera_municipio.csv", dtype={'municipio': str})
sitios = pd.read_csv(PATH_PROCESSED / "sitios.csv", dtype={'municipio': str})
pred = pred.merge(sitios[['sitio', 'anio_primera']], on='sitio', how='left')
obs = pred[pred['label'].notna()].reset_index(drop=True)
y = obs['label'].values
offset = logit(np.clip(obs['GLM'].values, 1e-4, 1 - 1e-4))
print(f"Sitios con resultado: {len(obs)}  |  tasa de hallazgo: {y.mean():.3f}")


In [ ]:
# ============================================================
# ZONAS OPERATIVAS
# ============================================================
obs['zona_municipio'] = obs['municipio']
periodo = np.where(obs['anio_primera'].isna(), 'sin año',
                   np.where(obs['anio_primera'] < 2010, 'antes de 2010', 'desde 2010'))
obs['zona_municipio_periodo'] = obs['municipio'] + ' | ' + periodo

# Fuente de la información: campo "Fuente de los datos" de las hojas de diligencias de las
# matrices de EQUITAS. Cada sitio toma la fuente más frecuente entre sus registros. No se
# imprimen los valores, porque algunos contienen nombres de personas.
fuentes = []
for archivo in sorted(glob.glob(PATH_MATRICES + "[!.]*.xlsx")):
    libro = pd.ExcelFile(archivo)
    hoja = [h for h in libro.sheet_names if 'Dilig' in h]
    if not hoja:
        continue
    d = libro.parse(hoja[0], header=1)
    col = [c for c in d.columns if str(c).startswith('Fuente de los datos')]
    if 'COD_REF_EQ' in d and col:
        fuentes.append(d[['COD_REF_EQ', col[0]]].set_axis(['COD_REF_EQ', 'fuente'], axis=1))
fuentes = pd.concat(fuentes).dropna(subset=['COD_REF_EQ'])
fuentes['COD_REF_EQ'] = fuentes['COD_REF_EQ'].astype(str).str.strip()
fuentes['fuente'] = fuentes['fuente'].astype(str).str.strip()
fuentes = fuentes[~fuentes['fuente'].str.lower().isin(['sin información', 'nan', ''])].drop_duplicates('COD_REF_EQ')

registros = pd.read_csv(PATH_PROCESSED / "registros_a_sitios.csv")
registros['COD_REF_EQ'] = registros['COD_REF_EQ'].astype(str).str.strip()
fuente_sitio = (registros.merge(fuentes, on='COD_REF_EQ').groupby('sitio')['fuente']
                .agg(lambda s: s.mode().iloc[0]))
obs['zona_fuente'] = obs['sitio'].map(fuente_sitio)
sin_fuente = obs['zona_fuente'].isna()
obs.loc[sin_fuente, 'zona_fuente'] = 'sitio ' + obs.loc[sin_fuente, 'sitio']  # zona propia
print(f"Sitios con fuente identificada: {(~sin_fuente).sum()} de {len(obs)}")

ZONAS = {'Municipio': 'zona_municipio', 'Municipio × período': 'zona_municipio_periodo',
         'Fuente de la información': 'zona_fuente'}
for nombre, col in ZONAS.items():
    tam = obs[col].value_counts()
    print(f"{nombre:<26} zonas: {len(tam):>3}  |  zonas con más de un sitio: {(tam > 1).sum():>3}  "
          f"|  sitios en ellas: {tam[tam > 1].sum()}")


In [ ]:
# ============================================================
# LOGIT DE EFECTOS ALEATORIOS CON OFFSET
# ============================================================
nodos, pesos = np.polynomial.hermite_e.hermegauss(N_CUADRATURA)  # integra contra exp(-x^2/2)
pesos = pesos / pesos.sum()

def log_verosimilitud(alpha, sigma, zonas):
    """Suma sobre zonas de log int prod_i Bernoulli(y_i; Lambda(alpha + l_i + u)) dN(u; 0, sigma^2)."""
    total = 0.0
    for idx in zonas:
        eta = alpha + offset[idx][:, None] + sigma * nodos[None, :]
        ll = (y[idx][:, None] * -np.logaddexp(0, -eta)
              + (1 - y[idx][:, None]) * -np.logaddexp(0, eta)).sum(axis=0)
        total += np.log(np.sum(pesos * np.exp(ll - ll.max()))) + ll.max()
    return total

def estimar(col_zona, offset_usado=None):
    global offset
    offset_base = offset
    if offset_usado is not None:
        offset = offset_usado
    try:
        return _estimar(col_zona)
    finally:
        offset = offset_base

def _estimar(col_zona):
    zonas = list(obs.groupby(col_zona).indices.values())
    nulo = optimize.minimize_scalar(lambda a: -log_verosimilitud(a, 0.0, zonas), bounds=(-5, 5), method='bounded')
    alt = optimize.minimize(lambda t: -log_verosimilitud(t[0], np.exp(t[1]), zonas),
                            x0=[nulo.x, np.log(0.5)], method='Nelder-Mead',
                            options={'xatol': 1e-6, 'fatol': 1e-8, 'maxiter': 4000})
    alpha, sigma = alt.x[0], np.exp(alt.x[1])
    ll0, ll1 = -nulo.fun, -alt.fun
    if ll1 < ll0:  # el máximo está en la frontera
        alpha, sigma, ll1 = nulo.x, 0.0, ll0
    lr = max(2 * (ll1 - ll0), 0.0)
    return {'alpha': alpha, 'sigma2': sigma**2, 'logv_nulo': ll0, 'logv_alt': ll1, 'LR': lr,
            'p_valor': 0.5 * stats.chi2.sf(lr, 1) if lr > 0 else 1.0}

# Offsets alternativos. El offset base usa los puntajes fuera de municipio, que están sesgados en
# contra de los municipios con alta tasa de hallazgo (al excluirlos baja la tasa de entrenamiento).
# Ese sesgo puede inflar la varianza entre municipios, de modo que se repite la prueba con el GLM
# estimado con todos los sitios y sin offset.
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
COVARIABLES = ['ndvi', 'altitud', 'pendiente', 'dist_hidrografia', 'dist_vias',
               'dist_centros_poblados', 'dist_grupos_armados']
glm_total = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
glm_total.fit(obs[COVARIABLES].values, y)
OFFSETS = {'GLM fuera de municipio (base)': offset,
           'GLM con todos los sitios': logit(np.clip(glm_total.predict_proba(obs[COVARIABLES].values)[:, 1], 1e-4, 1 - 1e-4)),
           'Sin offset': np.zeros(len(obs))}

resultados_prueba = pd.DataFrame([{'offset': o, 'zona': n, **estimar(c, off)}
                                  for o, off in OFFSETS.items() for n, c in ZONAS.items()])
print(resultados_prueba.round(4).to_string(index=False))


In [ ]:
# ============================================================
# ÍNDICE DE MORAN DE LOS RESIDUOS
# ============================================================
residuo = y - obs['GLM'].values
xy = np.c_[obs['lon'] * 111320 * np.cos(np.radians(obs['lat'])), obs['lat'] * 110540]
w = libpysal.weights.KNN(xy, k=5)
w.transform = 'r'
np.random.seed(SEMILLA)
moran = Moran(residuo, w, permutations=9999)
print(f"Índice de Moran (5 vecinos): I = {moran.I:.3f}  |  E[I] = {moran.EI:.3f}  |  "
      f"p (permutaciones) = {moran.p_sim:.4f}")

# Distancia típica a los 5 vecinos más cercanos
from scipy.spatial import cKDTree
dist, _ = cKDTree(xy).query(xy, k=6)
print(f"Distancia mediana al quinto vecino: {np.median(dist[:, 5]) / 1000:.1f} km")
resultados_prueba.to_csv(PATH_PROCESSED / "prueba_valor_informacion.csv", index=False)


## Simulación

- **Universo**: los 299 sitios candidatos, con costos $c_i = 1 + d_i$ (distancia a vías en km) y un presupuesto del 30% del costo total, común a todos los municipios para que la política pueda reasignarlo hacia las zonas donde aprende.
- **Zonas**: municipio × período, como en la prueba.
- **Verdad simulada**: $u_k \sim N(0, \sigma^2)$ y $G_i \sim \text{Bernoulli}(\Lambda(\alpha + \ell_i + u_{k(i)}))$, con $\ell_i$ el logit del puntaje del GLM y $\alpha$ fijado para que la probabilidad marginal promedio sea la tasa de hallazgo observada (0.70) con cualquier $\sigma^2$. Así, los escenarios difieren solo en la estructura compartida y no en el número esperado de hallazgos. Detección perfecta.
- **$\sigma^2 \in \{0, \hat\sigma^2, 2\hat\sigma^2\}$**, con $\hat\sigma^2$ estimado por municipio × período con el GLM estimado con todos los sitios como offset.
- **Políticas**: (i) portafolio estático: mochila con las probabilidades marginales iniciales; (ii) miope: en cada paso, el sitio de mayor $p_i(\mathcal D)/c_i$ que cabe en el presupuesto, con actualización bayesiana; (iii) política de la Sección 7.4: el sitio que maximiza $p_i(\mathcal D) + E_{y_i}[H_{-i}(b - c_i \mid \mathcal D \cup \{(i, y_i)\})]$. El valor de continuación $H$ se aproxima con la relajación lineal del problema de mochila (Dantzig, 1957), y en cada paso se evalúan los 30 sitios de mayor $p_i(\mathcal D)$ y los 30 de mayor $p_i(\mathcal D)/c_i$.
- La posterior de $u_k$ se calcula por cuadratura sobre los nodos de Gauss-Hermite, de modo que las probabilidades posteriores son coherentes (una martingala).


In [ ]:
# ============================================================
# MODELO DE SIMULACIÓN
# ============================================================
import os
sys.path.append('../src')
from politicas import mochila
import matplotlib.pyplot as plt

N_REPLICAS = 100
FRACCION_PRESUPUESTO = 0.3
N_CANDIDATOS_ENS = 30

cand = pred.copy()
cand['costo'] = 1 + cand['dist_vias'] / 1000
periodo_c = np.where(cand['anio_primera'].isna(), 'sin año',
                     np.where(cand['anio_primera'] < 2010, 'antes de 2010', 'desde 2010'))
zona_c = pd.factorize(cand['municipio'] + ' | ' + periodo_c)[0]
ell = logit(np.clip(cand['GLM'].values, 1e-4, 1 - 1e-4))
costo = cand['costo'].values
B = FRACCION_PRESUPUESTO * costo.sum()
N, K = len(cand), zona_c.max() + 1
miembros = [np.where(zona_c == k)[0] for k in range(K)]

sigma2_hat = resultados_prueba.query("offset == 'GLM con todos los sitios' and zona == 'Municipio × período'")['sigma2'].iloc[0]
ESCENARIOS = [0.0, sigma2_hat, 2 * sigma2_hat]
print(f"Sitios: {N}  |  zonas: {K}  |  presupuesto: {B:.1f}  |  sigma2_hat = {sigma2_hat:.3f}")

def calibrar_alpha(sigma):
    marg = lambda a: (pesos[None, :] * expit(a + ell[:, None] + sigma * nodos[None, :])).sum(axis=1).mean()
    return optimize.brentq(lambda a: marg(a) - y.mean(), -10, 10)

def prob_posterior(alpha, sigma, logw):
    """p_j(D) para todos los sitios, dados los log-pesos posteriores de cada zona sobre los nodos."""
    w = np.exp(logw - logw.max(axis=1, keepdims=True))
    w /= w.sum(axis=1, keepdims=True)
    return (w[zona_c] * expit(alpha + ell[:, None] + sigma * nodos[None, :])).sum(axis=1)

def relajacion_lineal(p, c, b):
    """Valor de la relajación lineal de la mochila: aproxima H(b | D)."""
    if b <= 0 or len(p) == 0:
        return 0.0
    orden = np.argsort(-p / c)
    acum = np.cumsum(c[orden])
    k = np.searchsorted(acum, b, side='right')
    valor = p[orden[:k]].sum()
    if k < len(p):
        valor += p[orden[k]] * (b - (acum[k - 1] if k > 0 else 0)) / c[orden[k]]
    return valor


In [ ]:
# ============================================================
# POLÍTICAS
# ============================================================
def simular(politica, alpha, sigma, G):
    """Devuelve la secuencia (costo acumulado, hallazgos acumulados) de una política."""
    disponible = np.ones(N, bool)
    logw = np.tile(np.log(pesos), (K, 1))
    b, gasto, hallazgos, traza = B, 0.0, 0, [(0.0, 0)]
    if politica == 'Portafolio estático':
        p0 = prob_posterior(alpha, sigma, logw)
        a = mochila(p0, costo, B).astype(bool)
        for i in np.where(a)[0][np.argsort(-(p0 / costo)[a])]:
            gasto += costo[i]; hallazgos += G[i]; traza.append((gasto, hallazgos))
        return traza
    while True:
        p = prob_posterior(alpha, sigma, logw)
        factibles = np.where(disponible & (costo <= b + 1e-9))[0]
        if len(factibles) == 0:
            break
        if politica == 'Miope (probabilidad por costo)':
            i = factibles[np.argmax(p[factibles] / costo[factibles])]
        else:  # Sección 7.4
            cands = np.union1d(factibles[np.argsort(-p[factibles])[:N_CANDIDATOS_ENS]],
                               factibles[np.argsort(-(p / costo)[factibles])[:N_CANDIDATOS_ENS]])
            mejor, i = -np.inf, None
            for j in cands:
                k = zona_c[j]
                resto = disponible.copy(); resto[j] = False
                valor = p[j]
                for y_j, prob_y in [(1, p[j]), (0, 1 - p[j])]:
                    lw = logw.copy()
                    eta = alpha + ell[j] + sigma * nodos
                    lw[k] += -np.logaddexp(0, -eta) if y_j else -np.logaddexp(0, eta)
                    p_y = p.copy()
                    p_y[miembros[k]] = prob_posterior(alpha, sigma, lw)[miembros[k]]
                    valor += prob_y * relajacion_lineal(p_y[resto], costo[resto], b - costo[j])
                if valor > mejor:
                    mejor, i = valor, j
        # intervenir i y actualizar la posterior de su zona
        eta = alpha + ell[i] + sigma * nodos
        logw[zona_c[i]] += -np.logaddexp(0, -eta) if G[i] else -np.logaddexp(0, eta)
        disponible[i] = False
        b -= costo[i]; gasto += costo[i]; hallazgos += G[i]
        traza.append((gasto, hallazgos))
    return traza


In [ ]:
# ============================================================
# SIMULACIÓN
# ============================================================
POLITICAS_SIM = ['Portafolio estático', 'Miope (probabilidad por costo)', 'Política de la Sección 7.4']
REJILLA_COSTO = np.linspace(0, B, 101)
rng = np.random.default_rng(SEMILLA)
filas, curvas = [], {}
for s2 in ESCENARIOS:
    sigma = np.sqrt(s2)
    alpha = calibrar_alpha(sigma)
    totales = {pol: [] for pol in POLITICAS_SIM}
    curvas_s2 = {pol: [] for pol in POLITICAS_SIM}
    for r in range(N_REPLICAS):
        u = rng.normal(0, sigma, K)
        G = (rng.random(N) < expit(alpha + ell + u[zona_c])).astype(int)
        for pol in POLITICAS_SIM:  # las tres políticas enfrentan la misma realización
            traza = np.array(simular(pol, alpha, sigma, G))
            totales[pol].append(traza[-1, 1])
            idx = np.searchsorted(traza[:, 0], REJILLA_COSTO, side='right') - 1
            curvas_s2[pol].append(traza[idx, 1])
    curvas[s2] = {pol: np.mean(v, axis=0) for pol, v in curvas_s2.items()}
    base = np.array(totales['Portafolio estático'])
    for pol in POLITICAS_SIM:
        t = np.array(totales[pol])
        filas.append({'sigma2': s2, 'politica': pol, 'hallazgos': t.mean(),
                      'ee': t.std(ddof=1) / np.sqrt(len(t)),
                      'dif_vs_estatico': (t - base).mean(),
                      'ee_dif': (t - base).std(ddof=1) / np.sqrt(len(t))})
    print(f"sigma2 = {s2:.2f} listo")

df_sim = pd.DataFrame(filas)
df_sim.to_csv(PATH_PROCESSED / "simulacion_valor_informacion.csv", index=False)
print(df_sim.round(3).to_string(index=False))


In [ ]:
# ============================================================
# FIGURA
# ============================================================
COLORES = {'Portafolio estático': '#5f5e5a', 'Miope (probabilidad por costo)': '#2a78d6',
           'Política de la Sección 7.4': '#eb6834'}
ESTILOS = {'Portafolio estático': '--', 'Miope (probabilidad por costo)': '-.',
           'Política de la Sección 7.4': '-'}
TINTA, TINTA_SEC, REJILLA = '#1f1f1e', '#5f5e5a', '#e4e3dd'
plt.rcParams.update({'font.family': 'serif', 'font.size': 9, 'axes.edgecolor': TINTA_SEC,
                     'axes.labelcolor': TINTA, 'xtick.color': TINTA_SEC, 'ytick.color': TINTA_SEC})
fig, ejes = plt.subplots(1, 3, figsize=(10, 3.6), sharey=True)
for ax, s2, letra in zip(ejes, ESCENARIOS, 'abc'):
    for pol in POLITICAS_SIM:
        ax.plot(REJILLA_COSTO, curvas[s2][pol], color=COLORES[pol], ls=ESTILOS[pol], lw=1.8, label=pol)
    ax.set_title(f'({letra}) $\\sigma^2 = {s2:.2f}$', loc='left', fontsize=9, color=TINTA)
    ax.set_xlabel('Costo acumulado')
    ax.grid(True, color=REJILLA, lw=0.6); ax.set_axisbelow(True)
    for lado in ['top', 'right']:
        ax.spines[lado].set_visible(False)
ejes[0].set_ylabel('Hallazgos acumulados')
manijas, etiquetas = ejes[0].get_legend_handles_labels()
fig.legend(manijas, etiquetas, frameon=False, fontsize=8, ncol=3, loc='lower center', bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.savefig("../data/images/simulacion_valor_informacion.pdf", bbox_inches='tight')
fig.savefig("../data/images/simulacion_valor_informacion.png", dpi=300, bbox_inches='tight')
print("✓ Figura guardada")
